# Transformer models for relevance and stance (Google Colab, T4 GPU)

This notebook scores the same (claim, post) pairs as `src/relevance.py` and `src/stance.py` with three transformer models:

1. **Multilingual sentence embeddings** (`intfloat/multilingual-e5-base`), no training: cosine similarity between the claim and the post.
2. **Zero-shot NLI** (`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`), no training: does the post entail "this text is about the claim" (relevance), and does it entail or contradict the claim itself (stance)?
3. **CamemBERT fine-tuned** (`almanach/camembert-base`) on (claim, post) pairs to predict relevance, trained and tested on the claim-grouped folds of `data/folds.csv`.

**Run it**
1. Runtime > Change runtime type > T4 GPU.
2. Runtime > Run all. When asked, upload `colab_inputs.zip` (posts_raw.csv, claims.csv, stance_labels.csv, folds.csv).
3. About 30 minutes later, `transformer_results.zip` downloads. Unzip it into `reports/transformers/`, then run `python src/relevance.py` and `python src/stance.py`: both add the transformer models to their tables.

In [ ]:
!pip -q install -U sentence-transformers  # torch and transformers are preinstalled on Colab
import os, re, time, zipfile
import numpy as np, pandas as pd, torch
from sklearn.metrics import average_precision_score, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none, select Runtime > Change runtime type > T4 GPU")

In [ ]:
INPUTS = ["posts_raw.csv", "claims.csv", "stance_labels.csv", "folds.csv"]
if not all(os.path.exists(f) for f in INPUTS):
    from google.colab import files
    for name in files.upload():  # choose colab_inputs.zip
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")
missing = [f for f in INPUTS if not os.path.exists(f)]
assert not missing, f"missing files: {missing}"

## Data

Same cleaning as `src/common.py`. Transformers read at most a few hundred tokens, so each post is cut to its first 300 words (title first).

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"u/\w+|r/\w+", "", text)
    text = re.sub(r"\[removed\]|\[deleted\]|&amp;#x200b;|&amp;|&gt;|&lt;", " ", text)
    text = re.sub(r"[^\w\s'\-àâçéèêëîïôùûüÿœæ.,!?;:]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

posts, claims = pd.read_csv("posts_raw.csv"), pd.read_csv("claims.csv")
labels, folds = pd.read_csv("stance_labels.csv"), pd.read_csv("folds.csv")
df = (posts.merge(labels[["post_id", "stance"]], on="post_id")
      .merge(claims[["claim_id", "claim", "verdict"]], on="claim_id")
      .sort_values("post_id").reset_index(drop=True))
df["passage"] = df["text"].map(clean_text).map(lambda t: " ".join(t.split()[:300]))
df["relevant"] = (df["stance"] != "off_topic").astype(int)
SEEDS = sorted(int(s) for s in folds["seed"].unique())
FOLD = {s: df[["post_id"]].merge(folds[folds["seed"] == s], on="post_id", how="left")["fold"].to_numpy() for s in SEEDS}
print(f"{len(df)} posts, {df['relevant'].sum()} relevant, seeds {SEEDS}")

def evaluate(scores, name):
    """Average precision on each claim-grouped test fold, as in src/relevance.py (rough check only)."""
    y, aps = df["relevant"].to_numpy(), []
    for s in SEEDS:
        score = scores[s] if isinstance(scores, dict) else scores
        for k in np.unique(FOLD[s]):
            m = FOLD[s] == k
            if y[m].sum():
                aps.append(average_precision_score(y[m], score[m]))
    print(f"{name}: average precision {100 * np.mean(aps):.1f}% ± {100 * np.std(aps):.1f}"
          f" (share of relevant posts: {100 * y.mean():.1f}%)")

results = []  # one table per model: post_id, model, seed (-1 = no training, same for every seed), score

## 1. Multilingual sentence embeddings, no training

The encoder turns the claim and the post into two vectors of 768 numbers trained so that texts with the same meaning get close vectors. The score is the cosine of the angle between them. e5 expects the prefixes `query:` and `passage:`.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-base", device=DEVICE)
encoder.max_seq_length = 512
claim_vecs = encoder.encode(["query: " + c for c in claims["claim"]], normalize_embeddings=True)
post_vecs = encoder.encode(["passage: " + p for p in df["passage"]], batch_size=32,
                           normalize_embeddings=True, show_progress_bar=True)
row = df["claim_id"].map({cid: i for i, cid in enumerate(claims["claim_id"])}).to_numpy()
e5_score = (post_vecs * claim_vecs[row]).sum(axis=1)
evaluate(e5_score, "Multilingual embeddings")
results.append(pd.DataFrame({"post_id": df["post_id"], "model": "Multilingual embeddings (e5), cosine",
                             "seed": -1, "score": e5_score}))
del encoder; torch.cuda.empty_cache()

## 2. Zero-shot NLI, no training

An NLI model reads a premise and a hypothesis together and outputs three probabilities: entailment, neutral, contradiction. Premise: the post.
- Relevance: hypothesis "Ce texte parle de l'affirmation : <claim>"; the score is the entailment probability.
- Stance: hypothesis = the claim itself; entailment -> supports, contradiction -> refutes, neutral -> discusses.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

NLI = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tok = AutoTokenizer.from_pretrained(NLI)
nli = AutoModelForSequenceClassification.from_pretrained(NLI).to(DEVICE).eval()
LABEL = {v.lower(): k for k, v in nli.config.id2label.items()}  # entailment, neutral, contradiction

@torch.no_grad()
def nli_probs(premises, hypotheses, batch_size=32, max_length=384):
    out = []
    for i in range(0, len(premises), batch_size):
        enc = nli_tok(premises[i:i + batch_size], hypotheses[i:i + batch_size], truncation="only_first",
                      max_length=max_length, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = nli(**enc).logits.float()
        if not torch.isfinite(logits).all():  # half-precision overflow: redo this batch in full precision
            logits = nli(**enc).logits.float()
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out)

premises = df["passage"].tolist()
topic = nli_probs(premises, ["Ce texte parle de l'affirmation : " + c for c in df["claim"]])
nli_relevance = topic[:, LABEL["entailment"]]
evaluate(nli_relevance, "Zero-shot NLI")
results.append(pd.DataFrame({"post_id": df["post_id"], "model": "Zero-shot NLI (mDeBERTa)",
                             "seed": -1, "score": nli_relevance}))

stance = nli_probs(premises, df["claim"].tolist())
stance_nli = pd.DataFrame({"post_id": df["post_id"], "p_entailment": stance[:, LABEL["entailment"]],
                           "p_neutral": stance[:, LABEL["neutral"]], "p_contradiction": stance[:, LABEL["contradiction"]]})
rel = df["relevant"] == 1
pred = np.array(["supports", "discusses", "refutes"])[
    stance_nli[["p_entailment", "p_neutral", "p_contradiction"]].to_numpy()[rel].argmax(axis=1)]
print("Zero-shot stance on relevant posts, macro-F1:",
      round(100 * f1_score(df.loc[rel, "stance"], pred, average="macro"), 1), "%")
del nli; torch.cuda.empty_cache()

## 3. CamemBERT fine-tuned on (claim, post) pairs

CamemBERT is a BERT-type model pre-trained on French text. It reads `<s> claim </s></s> post </s>` and a small classification layer on top outputs P(relevant).
For each seed and each of the 5 folds, a fresh model is fine-tuned on the claims of 4 folds and scores the posts of the 5th, whose claims it has never seen.
Relevant posts are rare (6.6%), so their errors weigh more in the loss.

In [ ]:
from transformers import get_linear_schedule_with_warmup

CAMEMBERT = "almanach/camembert-base"
MAX_LEN, EPOCHS, BATCH, LR = 256, 3, 16, 2e-5
tok = AutoTokenizer.from_pretrained(CAMEMBERT)
ENC = tok(df["claim"].tolist(), df["passage"].tolist(), truncation="only_second", max_length=MAX_LEN)
Y = df["relevant"].to_numpy()

def batches(idx, rng=None):
    if rng is not None:
        idx = rng.permutation(idx)
    for i in range(0, len(idx), BATCH):
        b = idx[i:i + BATCH]
        x = tok.pad({k: [ENC[k][j] for j in b] for k in ("input_ids", "attention_mask")}, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in x.items()}, torch.tensor(Y[b], device=DEVICE)

def train_and_score(train_idx, test_idx, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = AutoModelForSequenceClassification.from_pretrained(CAMEMBERT, num_labels=2).to(DEVICE)
    pos = Y[train_idx].mean()
    weight = torch.tensor([1.0, float(np.sqrt((1 - pos) / pos))], device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(train_idx) / BATCH))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    model.train()
    for _ in range(EPOCHS):
        for x, y in batches(train_idx, rng):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                logits = model(**x).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), y, weight=weight)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
    model.eval()
    scores = []
    with torch.no_grad():
        for x, _ in batches(test_idx):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                scores.append(torch.softmax(model(**x).logits.float(), dim=-1)[:, 1].cpu().numpy())
    del model, opt
    torch.cuda.empty_cache()
    return np.concatenate(scores)

ft = {}
for seed in SEEDS:
    ft[seed] = np.zeros(len(df))
    for k in np.unique(FOLD[seed]):
        t0 = time.time()
        test_idx, train_idx = np.where(FOLD[seed] == k)[0], np.where(FOLD[seed] != k)[0]
        ft[seed][test_idx] = train_and_score(train_idx, test_idx, seed)
        print(f"seed {seed}, fold {k}: {time.time() - t0:.0f} s")
    results.append(pd.DataFrame({"post_id": df["post_id"], "model": "CamemBERT fine-tuned on (claim, post) pairs",
                                 "seed": seed, "score": ft[seed]}))
evaluate(ft, "CamemBERT fine-tuned")

## Save the scores

`transformer_results.zip` holds `relevance_scores.csv` and `stance_nli.csv` (post IDs and scores, no text).

In [ ]:
os.makedirs("transformers", exist_ok=True)
pd.concat(results).round({"score": 5}).to_csv("transformers/relevance_scores.csv", index=False)
stance_nli.round(5).to_csv("transformers/stance_nli.csv", index=False)
with zipfile.ZipFile("transformer_results.zip", "w") as z:
    for f in ("relevance_scores.csv", "stance_nli.csv"):
        z.write(f"transformers/{f}", f)
try:
    from google.colab import files
    files.download("transformer_results.zip")
except ImportError:
    print("Saved transformer_results.zip")